# Debugging Missings Pre & Post County Formations Estimation Merge

## Setup

In [59]:
# Packages
import pandas as pd

# Loading Datasets
state_forms = pd.read_csv('../../data_clean/annualized_state_formations.csv',
                           dtype={'full_fips': str, 'state_fips': str, 'county_fips': str})
state_apps = pd.read_csv('../../data_clean/state_apps_annual.csv',
                         dtype={'full_fips': str, 'state_fips': str, 'county_fips': str})
county_apps = pd.read_csv('../../data_clean/annual_county_applications.csv',
                          dtype={'full_fips': str, 'state_fips': str, 'county_fips': str})

## Pre-Merge Missing Data
Getting dataframes of missing observations from all three datasets used in the calculation PRE-MERGE to compare with the missing observations in the POST-MERGE dataset to determine true missing vs merge issue.

In [60]:
# State Formations
sf_og_missing = state_forms[state_forms['state_formations'].isna()]
sf_og_missing.to_csv('sf_og_missing.csv') 

# State Applications
sa_og_missing = state_apps[state_apps['state_apps'].isna()]
sa_og_missing.to_csv('sa_og_missing.csv')

# County Applications
ca_og_missing = county_apps[county_apps['county_apps'].isna()]
ca_og_missing.to_csv('ca_og_missing.csv')

## Updated county_formations_estimation.py file
All actual code edits happen in the .py file. I copy-paste the updated file (minus the setup section) here and comment out the final line. 

In [61]:

# #################################
# MERGING DATASETS

# Merging state formations and state applications
state = pd.merge(state_forms, state_apps, on=['year', 'state_fips', 'STATE', 'STATE_NAME'])
assert state.groupby(['state_fips', 'year']).size().eq(1).all()

# Merging state dataframe with county applications
est = pd.merge(county_apps, state, on=['year', 'state_fips', 'STATE'], validate='many_to_one')
assert est.groupby(['full_fips', 'year']).size().eq(1).all()

# Merge validation
assert est['state_apps'].notna().all()

# Columns Cleanup
est = est.rename(columns={'County': 'county'})
est = est.reindex(columns = ['STATE', 'year', 'county', 'full_fips', 'state_formations', 'state_apps', 'county_apps', 'state_fips', 'county_fips', 'STATE_NAME'])


# #################################
# ESTIMATING COUNTY FORMATIONS

# Correcting variable types
est['county_apps'] = pd.to_numeric(est['county_apps'], errors='coerce')

# Calculating estimated county formations
est['county_forms_est'] = (est['state_formations']/est['state_apps']) * est['county_apps']


# #################################
# SAVING DATASET
# est.to_csv('data_clean/county_formations_estimation.csv', index=False)

## Investigation Code:
Debugging shenanigans start here!

Abbreviations Guide:
* sf = state formations
* sa = state applications
* ca = county applications
* og = original, meaning pre-merge data

In [62]:
# FINDING POST-MERGE MISSING DATA IN EACH VARIABLE
sf_missing = est[est['state_formations'].isna()] # Missings in state formations
sa_missing = est[est['state_apps'].isna()] # Missings in state applications
ca_missing = est[est['county_apps'].isna()] # Missings in county applications

**State Formation (SF) Missing Pre/Post Merge Comparison**

Code below, results here!
Because the sf_missing_problematic dataframe is coming up empty, all of the missing state formations observations in post-merge dataset are all from the original 47 missing obs in the pre-merge dataset (just expanded to the county level)

In [63]:
# Merging sf missing dataframes to separate out true missing (47) vs the large remainder of missings in the final data
sf_pre_post_missing = pd.merge(sf_missing, sf_og_missing, on=['state_fips', 'year'], indicator=True)

sf_missing_problematic = sf_pre_post_missing[sf_pre_post_missing['_merge']!= 'both']

**State Application (SA) Missing Pre/Post Merge Comparison**

All is well with state applications! NO MISSING PRE OR POST! :)



**County Application (CA) Missing Investigation**

Problem: 
* No missing observations in clean CA dataset (ca_og_missing is empty)
* 227 mising observations post merge (227 rows in ca_missing)

Discoveries from code below:
* the 227 missing observations are missing in both the pre and post merge data
* the original download contains significant suppressed data that was not noticed previously because they are blank strings ' '




In [73]:
# Are the missing 227 observations in the post-merge dataset also missing in the pre-merge, clean CA dataset?
ca_missing_check = ca_missing[
    ['full_fips', 'year', 'STATE', 'county_apps']
].merge(
    county_apps[['full_fips', 'year', 'county_apps']],
    on=['full_fips', 'year'],
    how='left',
    suffixes=('_post', '_original'),
    indicator=True,
    validate='one_to_one'
)

ca_missing_check['_merge'].value_counts()

_merge
both          227
left_only       0
right_only      0
Name: count, dtype: int64

In [74]:
# How many blank strings are in the cleaned dataset?
county_apps['county_apps'].map(type).value_counts()
county_apps.loc[
    ~county_apps['county_apps'].isna()
    & pd.to_numeric(county_apps['county_apps'], errors='coerce').isna(),
    'county_apps'
].map(repr).value_counts()


county_apps
' '    263
Name: count, dtype: int64

In [83]:
# INVESTIGATING WHERE THE MISSING DATA IS
ca_blank = county_apps.loc[
    county_apps['county_apps'].notna()
    & pd.to_numeric(county_apps['county_apps'], errors='coerce').isna()
].copy()

print(ca_blank.shape)

print(ca_blank[['full_fips', 'STATE', 'year', 'county_apps']].head(20))
print(ca_blank.groupby('year').size())
print(ca_blank.groupby('STATE').size().sort_values(ascending=False))
for x in ca_blank['county_apps'].unique():
    print(repr(x), len(x), [ord(c) for c in x])

(263, 7)
     full_fips STATE  year county_apps
1440     02063    AK  2005            
1441     02063    AK  2006            
1442     02063    AK  2007            
1443     02063    AK  2008            
1444     02063    AK  2009            
1445     02063    AK  2010            
1446     02063    AK  2011            
1447     02063    AK  2012            
1448     02063    AK  2013            
1449     02063    AK  2014            
1450     02063    AK  2015            
1451     02063    AK  2016            
1452     02063    AK  2017            
1453     02063    AK  2018            
1454     02063    AK  2019            
1460     02066    AK  2005            
1461     02066    AK  2006            
1462     02066    AK  2007            
1463     02066    AK  2008            
1464     02066    AK  2009            
year
2005    13
2006    13
2007    13
2008    13
2009    13
2010    13
2011    13
2012    13
2013    13
2014    14
2015    14
2016    14
2017    14
2018    14
2019    14
20

In [87]:
# Investigating where the missing CA data is part 2:
ca_blank.groupby(
    ['STATE', 'full_fips']
).agg(
    n_blank=('year', 'size'),
    first_blank=('year', 'min'),
    last_blank=('year', 'max')
).reset_index()

ca_missing.to_csv('ca_missing.csv')
